# Qdrant

Qdrant 的每一筆資料, 都稱之為一個 point

- RDB → 一列 row
- DynamoDB → 一個 item
- Neo4j → 一個 node
- Qdrant → 一個 point

一個 point 通常包含了:

```
{
  "id": 1,
  "vector": [0.12, 0.98, ...],
  "payload": {
    "title": "hello"
  }
}
```

| RDB      | Qdrant                       |
| -------- | ---------------------------- |
| Database | Qdrant instance              |
| Table    | Collection                   |
| Row      | Point                        |
| Column   | Payload fields               |
| Index    | Vector index / payload index |


qdrant 針對特定 payload 可以建立 index, 而這些 index type 基本上會有:

- keyword  字串且要做精確篩選
- text     全文字串搜尋
- integer
- float
- bool
- datetime
- uuid
- geo

https://api.qdrant.tech/api-reference/indexes/create-field-index


## vector type

- dense vector 用於 語意搜尋/相似度搜尋, 最廣泛用來使用 vector search 的類型
- sparse vector 用來做 keyword search
  - 同 dense vector 一樣是高維度, 然而多數欄位都是 0.
  - 常見方法有 
    - Statistical methods
      - BM25
      - TF-IDF/BoW
    - Sparse neural encoders
      - SPLADE/SPLADE++
      - DeepImpact/UniCOIL
- multivectors 
  - matrix of dense vectors per point
- named vectors
  - 由於 Qdrant 的 Point 有三個主要屬性: id, vector, payload. vector 基本上都是 [], 但也支援 named vectors, 型別為 {}
  - 直接看下面範例

Point 儲存時, 可以同時儲存它的 `Sparse Vector + Dense Vector`. 也可以儲存 `一個 Sparse Vector + 多個 Dense Vectors`, 或者 `一個 Sparse Vector + 多個 Multivector`.

```py
client.create_collection(
  collection_name = "demo",
  vectors_config = {
    "image": models.VectorParams(size=4, distance=models.Distance.COSINE),
    "text": models.VectorParams(size=5, distance=models.Distance.COSINE),
  },
  sparse_vectors_config={
    "text-sparse": models.SparseVectorParams()
  }
)

client.upsert(
    collection_name="demo",
    points=[
        models.PointStruct(
            id=1,
            vector={
                "image": [.9, .1, .1, .2],
                "text": [.4, .7, .1, .8, .1],
                "text-sparse": {
                    "indices": [1, 3, 5, 7],
                    "values": [.1, .2, .3, .4],
                },
            },
        ),
    ],
)
```


In [ ]:
## Payload types

- Integer
- Float
- Bool
- Keyword
- Geo
- Datetime
- UUID

Scalar - Numerical, Bool. 用於篩選 price, rating, in_stock, ...
Categorical - Category labels. 用於篩選 category, brand, ...
Geolocation - Latitude / Longitude. 用於篩選經緯度
Timestamps - Date and Time. 篩選時間
Arrays - multiple values
Nested Objects - JSON-like ata

Logical Filters
    - Logical Operators : AND / OR / NOT
    - must, should, must_not, ...

Match
    - Match Any (OR)
    - Match (Exact Value)
    - Match Except (NOT IN)
    - Range

## 資料量

- 一個 1536 維的 Float 32 Point 資料量大概要 6KB (官方給的粗估). 因此 100萬個 Points 大概就需要 6GB Disk